# Correlation matrix for longitudinal modelling

**z-gain** and **thrivelines** need an age-to-age **correlation matrix** per region (how normative z-scores correlate between ages). In PCNtoolkit you can estimate that matrix from a longitudinal reference cohort with predicted z-scores.

The **z-gain** score and **thrivelines** are defined in [Bayer et al., 2026](https://arxiv.org/abs/2601.07591). This notebook covers **estimation and inspection** of the matrix. Loading a precomputed matrix and scoring subjects is in [tutorial 14](14_longitudinal_modelling.ipynb).


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pcntoolkit import BLR, CorrelationMatrix, NormData, NormativeModel
from pcntoolkit.math_functions.basis_function import BsplineBasisFunction

COVARIATE = "age"
DEMO_REGION = "rh_S_front_sup"


def find_data_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "pcntoolkit_resources" / "data"
        if (candidate / "fcon1000.csv").exists() and (candidate / "LNM_data.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find pcntoolkit_resources/data with fcon1000.csv and LNM_data.csv"
    )


def tidy_region_names(df: pd.DataFrame) -> pd.DataFrame:
    metadata = {"sub_id", "age", "sex", "site", "visit", "group"}
    return df.rename(
        columns={
            c: c.replace("&", "_and_").replace("_thickness", "")
            for c in df.columns
            if c not in metadata
        }
    )


data_dir = find_data_dir()


## 1. Reference model and longitudinal controls

Same setup as tutorial 14: fit BLR on **fcon1000**, transfer to LNM **controls**, predict z-scores on a control subset used for matrix estimation.


In [ ]:
reference_df = tidy_region_names(pd.read_csv(data_dir / "fcon1000.csv"))
lnm_df = tidy_region_names(pd.read_csv(data_dir / "LNM_data.csv"))

if DEMO_REGION not in reference_df.columns:
    DEMO_REGION = [
        c for c in reference_df.columns if c not in {"sub_id", "age", "sex", "site"}
    ][0]
response_vars = [DEMO_REGION]

reference_data = NormData.from_dataframe(
    name="reference",
    dataframe=reference_df,
    covariates=["age"],
    batch_effects=["site", "sex"],
    response_vars=response_vars,
    subject_ids="sub_id",
)

fcon1000_model = NormativeModel(
    BLR(
        basis_function_mean=BsplineBasisFunction(degree=3, nknots=5),
        warp_name="warpsinharcsinh",
    ),
    inscaler="standardize",
    outscaler="standardize",
    savemodel=False,
    saveresults=False,
)
fcon1000_model.fit(reference_data)

control_df = lnm_df[lnm_df["group"] == "control"]
control_data = NormData.from_dataframe(
    name="control",
    dataframe=control_df,
    covariates=["age"],
    batch_effects=["site", "sex"],
    response_vars=response_vars,
    subject_ids="sub_id",
    visits="visit",
)
control_adapt, control_test = control_data.train_test_split(splits=[0.2, 0.8])
lnm_model = fcon1000_model.transfer(control_adapt)
lnm_model.predict(control_test)

print("Control subjects (test split):", len(np.unique(control_test.get_subject_ids())))


## 2. Compute the correlation matrix

`CorrelationMatrix.compute()` needs longitudinal data with **`Z`** and **`Yhat`**. It returns an object with `matrix`, `bandwidth`, `n_subjects`, and `estimated_range`.

> The public LNM demo is small. Check `estimated_range` and the heatmap before trusting extrapolated ages.


In [ ]:
corr = CorrelationMatrix.compute(
    control_test,
    bandwidth=3,
    covariate=COVARIATE,
)

print("bandwidth:", corr.bandwidth)
print("n_subjects:", corr.n_subjects)
print("estimated_range:", corr.estimated_range)


## 3. Inspect the matrix


In [ ]:
region = response_vars[0]
R = corr.matrix.sel(response_vars=region)
age_lo, age_hi = corr.estimated_range
if age_lo is None or age_hi is None:
    age_lo, age_hi = 18, 50
R = R.sel(age_1=slice(age_lo, age_hi), age_2=slice(age_lo, age_hi))

plt.figure(figsize=(5, 4))
plt.imshow(
    R.values[0],
    origin="lower",
    extent=[age_lo, age_hi, age_lo, age_hi],
    vmin=-1,
    vmax=1,
    cmap="coolwarm",
)
plt.colorbar(label="correlation")
plt.xlabel("age (years)")
plt.ylabel("age (years)")
plt.title(f"Age-to-age z correlations — {region}")
plt.show()


## 4. Next steps

Pass `corr` to `ZGainScore(lnm_model, corr)` for z-gain scores and thrivelines. See [tutorial 14](14_longitudinal_modelling.ipynb).

Today you can **load** velocity-style pickles with `CorrelationMatrix.load()`. Native **save/load** for matrices estimated in PCNtoolkit is not implemented yet (placeholders below).


## 5. Save and load (placeholders)

PCNtoolkit will define a storage format for `CorrelationMatrix.save()` and a matching loader. Until then, use `load()` only for the velocity pickle layout documented in the API.


In [ ]:
# TODO: save correlation matrix once CorrelationMatrix.save() is implemented.
# corr.save("pcntoolkit_resources/data/my_correlation_matrix.pkl")


In [ ]:
# TODO: reload a matrix saved by PCNtoolkit (not the velocity pickle layout).
# corr_loaded = CorrelationMatrix.load("pcntoolkit_resources/data/my_correlation_matrix.pkl")


In [ ]:
# Today: load precomputed velocity pickles (see tutorial 14).
# corr_pickle = CorrelationMatrix.load("pcntoolkit_resources/velocity/.../Velocity/R.pkl")
